# LangChain: Agents

An **agent** is an LLM that decides *which tool to call* based on the user's question, calls it, observes the result, and repeats until it has a final answer.

```
User question
     │
     ▼
  ┌──────┐  tool call  ┌─────────────┐
  │ LLM  │ ──────────► │    Tool     │
  │agent │ ◄────────── │ (search,    │
  └──────┘  result     │  math, ...) │
     │                 └─────────────┘
     ▼ (repeat or finish)
  Final answer
```

## Outline

1. **Built-in tools** — calculator and Wikipedia search
2. **Python executor tool** — let the agent write and run code
3. **Custom tools** — define your own with the `@tool` decorator

---
> **API used:** `langgraph.prebuilt.create_react_agent` (LangGraph ≥ 1.0).  
> The old `initialize_agent` / `AgentType` / `create_python_agent` APIs are removed in LangChain 1.x.

In [1]:
import warnings
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
_ = load_dotenv(find_dotenv())

## Setup

In [2]:
LLM_MODEL = "gpt-3.5-turbo"

---
## Part 1 — Built-in Tools: Calculator and Wikipedia

We give the agent two tools:
- **calculator** — evaluates mathematical expressions using Python's `math` module
- **wikipedia** — searches Wikipedia and returns a short summary

The agent decides on its own which tool (if any) to use for each question.

### 1a. Define the Tools

In [4]:
# pip install wikipedia  (already installed in this venv)
import math
import wikipedia.wikipedia as _wiki

# The wikipedia package hardcodes http:// which returns 403 from Wikipedia's API.
# Patching to https:// here prevents JSONDecodeError when the agent calls the tool.
_wiki.API_URL = "https://en.wikipedia.org/w/api.php"

from langchain_openai import ChatOpenAI
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression.
    Input must be a valid Python math expression, e.g. '25 * 0.25' or 'sqrt(16)'.
    Supports all functions from Python's math module.
    """
    try:
        result = eval(expression, {"__builtins__": {}}, vars(math))
        return str(result)
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
)

tools = [calculator, wikipedia_tool]

print("Tools registered:")
for t in tools:
    print(f"  \u2022 {t.name}: {t.description[:70]}...")


Tools registered:
  • calculator: Evaluates a mathematical expression.
Input must be a valid Python math...
  • wikipedia: A wrapper around Wikipedia. Useful for when you need to answer general...


### 1b. Create the Agent

`create_react_agent` wires the LLM and tools into a ReAct loop (Reason + Act).  
The agent outputs a list of messages; we extract the last one as the final answer.

In [5]:
llm = ChatOpenAI(temperature=0, model=LLM_MODEL)
agent = create_react_agent(llm, tools)


def run_agent(question: str) -> str:
    """Invoke the agent and return the final answer as a plain string."""
    result = agent.invoke({"messages": [("human", question)]})
    return result["messages"][-1].content

### 1c. Math Example

In [6]:
answer = run_agent("What is 25% of 300?")
print(answer)

25% of 300 is 75.


### 1d. Wikipedia Example

In [7]:
question = (
    "Tom M. Mitchell is an American computer scientist and the Founders University "
    "Professor at Carnegie Mellon University (CMU). What book did he write?"
)
answer = run_agent(question)
print(answer)

Tom M. Mitchell wrote the book "Machine Learning."


### 1e. Inspect the ReAct Loop (Debug Mode)

Setting `langchain.debug = True` prints each step the agent takes:  
the LLM's reasoning, which tool it chose, the tool's output, and the final synthesis.

This is useful when the agent gives an unexpected answer.

In [8]:
import langchain

langchain.debug = True
run_agent("What is 25% of 300?")
langchain.debug = False

---
## Part 2 — Python Executor Tool

We give the agent the ability to write and execute arbitrary Python code.  
This is powerful for tasks like sorting, data manipulation, or calculations that are hard to express as a formula.

> **Security note:** `exec()` runs arbitrary code. Only use this in trusted, sandboxed environments.

In [9]:
import io
import contextlib


@tool
def python_executor(code: str) -> str:
    """Executes a Python code snippet and returns its printed output.
    Use this to perform computations, sorting, or data manipulation.
    The code must use print() to produce output.
    """
    output = io.StringIO()
    try:
        with contextlib.redirect_stdout(output):
            exec(code, {"__builtins__": __builtins__})  # noqa: S102
        return output.getvalue() or "(no output)"
    except Exception as e:
        return f"Error: {e}"


python_agent = create_react_agent(llm, [python_executor])


def run_python_agent(question: str) -> str:
    result = python_agent.invoke({"messages": [("human", question)]})
    return result["messages"][-1].content

In [10]:
customer_list = [
    ["Harrison", "Chase"],
    ["Lang", "Chain"],
    ["Dolly", "Too"],
    ["Elle", "Elem"],
    ["Geoff", "Fusion"],
    ["Trance", "Former"],
    ["Jen", "Ayai"],
]

In [11]:
answer = run_python_agent(
    f"Sort these customers by last name, then first name, and print the result: {customer_list}"
)
print(answer)

The customers sorted by last name, then first name are:

1. ['Chase', 'Harrison']
2. ['Chain', 'Lang']
3. ['Too', 'Dolly']
4. ['Elem', 'Elle']
5. ['Fusion', 'Geoff']
6. ['Former', 'Trance']
7. ['Ayai', 'Jen']


---
## Part 3 — Custom Tools with `@tool`

Any Python function decorated with `@tool` becomes an agent tool.  
The **docstring** is what the LLM reads to decide when and how to use the tool — write it carefully.

Rules for good tool docstrings:
- Describe *when* to call the tool (not just *what* it does)
- Specify the expected input format explicitly
- Keep it concise — the LLM reads this at every reasoning step

In [12]:
from datetime import date


@tool
def get_today() -> str:
    """Returns today's date in ISO format (YYYY-MM-DD).
    Use this whenever the user asks about the current date or needs to know today's date.
    Takes no input.
    """
    return str(date.today())


print(f"Tool name:        {get_today.name}")
print(f"Tool description: {get_today.description}")

Tool name:        get_today
Tool description: Returns today's date in ISO format (YYYY-MM-DD).
Use this whenever the user asks about the current date or needs to know today's date.
Takes no input.


### Combine All Tools into One Agent

In [13]:
full_agent = create_react_agent(llm, [calculator, wikipedia_tool, get_today])


def run_full_agent(question: str) -> str:
    result = full_agent.invoke({"messages": [("human", question)]})
    return result["messages"][-1].content

In [14]:
# The agent should call get_today() rather than guessing
answer = run_full_agent("What is today's date?")
print(answer)

Today's date is 2026-08-02.


### Multi-Tool Example

A question that requires both Wikipedia lookup and the calculator — the agent chains both tools in one run.

In [15]:
answer = run_full_agent(
    "According to Wikipedia, how many days are in a Martian year? "
    "Then calculate how many Earth days that is if a Martian sol is 1.027 Earth days."
)
print(answer)

According to Wikipedia, a Martian year is approximately 687 Earth days. 

Calculating the equivalent in Earth days, if a Martian sol is 1.027 Earth days:
687 Martian days * 1.027 Earth days = 705.549 Earth days

Therefore, a Martian year is approximately 705.549 Earth days.


---
## Summary

| Concept | What we did |
|---|---|
| Agent loop | `create_react_agent(llm, tools)` from `langgraph.prebuilt` |
| Built-in tools | `calculator` (`@tool` + `eval`), `WikipediaQueryRun` |
| Python execution | `python_executor` (`@tool` + `exec` + stdout capture) |
| Custom tool | `get_today` — returns today's date |
| Debug | `langchain.debug = True` prints every reasoning step |

**Key rule:** The LLM reads the tool's **docstring** to decide when to call it. A clear, specific docstring is the most important thing you can write when defining a tool.

**Next steps:**
- Add memory to the agent so it can hold a multi-turn conversation (see `checkpointer` param in `create_react_agent`)
- Build tools that call external APIs or query a database
- Explore `langgraph` for more complex multi-agent workflows